In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src.baseline import PopularityBaseline
from src.evaluation import ndcg_at_k, precision_at_k, recall_at_k

In [2]:
input_dir = Path("../data")

train = pd.read_csv(input_dir / "processed/train.csv")
test = pd.read_csv(input_dir / "processed/test.csv")
movies = pd.read_csv(input_dir / "ml-latest-small/movies.csv")

In [3]:
model = PopularityBaseline(train)
k = 10
predict = model.recommend_top_n(n=k)
titles = movies.set_index("movieId").loc[predict]["title"].tolist()
print(predict)
titles

[318, 858, 2959, 260, 50, 1221, 1213, 1197, 750, 527]


['Shawshank Redemption, The (1994)',
 'Godfather, The (1972)',
 'Fight Club (1999)',
 'Star Wars: Episode IV - A New Hope (1977)',
 'Usual Suspects, The (1995)',
 'Godfather: Part II, The (1974)',
 'Goodfellas (1990)',
 'Princess Bride, The (1987)',
 'Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)',
 "Schindler's List (1993)"]

In [4]:
test["relevant"] = test["movieId"].apply(lambda x: x if isinstance(x, (set, list)) else {x})

test[f"precision@{k}"] = test.apply(lambda row: precision_at_k(predict, row["relevant"], k), axis=1)

test[f"recall@{k}"] = test.apply(lambda row: recall_at_k(predict, row["relevant"], k), axis=1)

test[f"ndcg@{k}"] = test.apply(lambda row: ndcg_at_k(predict, row["relevant"], k), axis=1)

mean_precision = test[f"precision@{k}"].mean()
mean_recall = test[f"recall@{k}"].mean()
mean_ndcg = test[f"ndcg@{k}"].mean()

print(f"Mean Precision@{k}: {mean_precision:.4f}")
print(f"Mean Recall@{k}:    {mean_recall:.4f}")
print(f"Mean nDCG@{k}:      {mean_ndcg:.4f}")

Mean Precision@10: 0.0021
Mean Recall@10:    0.0213
Mean nDCG@10:      0.0106
